## Objectives
1. Learn how `AlphaSimR` simulates traits that are affected by genotype by
environment interaction.  
2. Analyze a multi-environment trial using the "compound symmetric" covariance
structure.  
3. Look at the impact of sub-dividing the "world" into "mega-environments" on
the within-mega-environment main genetic effect.  

## R scripts  

### Script setup
Install packages set the random seed  

In [ ]:
#Loading libraries
req_packages<-c("ggplot2", "lme4","AlphaSimR")

for(i in c(1:length(req_packages))){
  if (!require(req_packages[i], character.only = TRUE)){
   install.packages(req_packages[i])
  }
}
random_seed <- 456789
set.seed(random_seed)

### GxE

#### Simulate GxE data in AlphaSimR
The function for an additive trait affected by GxE is `addTraitAG`. The kind of
environment and therefore the GxE deviations is set by specifying a value, `p`,
from the unit uniform, with the target environment represented by 0.5 and `0 < p
< 1` at opposite ends of a spectrum. Each time you call `setPheno` you have to
specify this value for the environment from which you are getting phenotypes.
`AlphaSimR` translates this `p` to a GxE deviation as follows (full details
here: [pdf about traits in AlphaSimR](./AlphaSimR_Traits_2025.pdf))
The GxE deviation for individual $j$ in environment $i$ is  
$$ge_{ij} = w_ib_j$$  
where $w_i$ is effectively a loading for environment $i$ calculated as
`qnorm(p)`. Thus, $w_i$ has a variance of 1. $b_j$ is a score for individual $j$
that is an unobserved trait of individual $j$ that is additive and has a
variance equal to the GxE deviation variance $\sigma^2_{ge}$.  
With this parameterization of the GxE deviation, the covariance between deviations in environments $i$ and $i'$ is $w_iw_{i'}\sigma^2_{ge}$.  Denoting the genotypic value for individual $j$ in environment $i$ as
$$g_{ij} = g_j + ge_{ij}$$
The genetic variance within environment $i$ is  
var($g_{ij}$) = $\sigma^2_g + w^2_i\sigma^2_{ge}$  
and the genetic covariance between environment $i$ and environment $i'$ is  
cov($g_{ij}$, $g_{ij'}$) = $\sigma^2_g + w_iw_{i'}\sigma^2_{ge}$  

In [ ]:
nChr <- 7
segSites <- 40
nQTL <- 40 # per chromosome
# Create a new population of founders
nFounders <- 50
founderHaps <- AlphaSimR::runMacs(nInd=nFounders, nChr=nChr, segSites=segSites)
SP <- AlphaSimR::SimParam$new(founderHaps)
SP$addTraitAG(nQtlPerChr=nQTL, mean=0, var=1, varGxE=1, varEnv=0)
# Create a new population of founders
founders <- AlphaSimR::newPop(founderHaps, simParam=SP)

### Code to conduct a "multi-environment trial"

In [ ]:
# pop is an AlphaSimR population
# envPvals is a matrix with nTrials rows and two columns
# Each row specifies the range of p-values permissible for the trial
# To specify a particular value for that trial, set min and max to same value
# varE is the error variance for each trial
conductMET <- function(pop, envPvals=NULL, nReps=2, varE=NULL){
  if (is.null(envPvals)){ # Two environments with any loading
    envPvals <- matrix(c(0.05,0.05,0.95,0.95), nrow=2)
  }
  if (is.null(varE)) varE <- 1
  nTrials <- nrow(envPvals)
  allPheno <- allGenoVal <- NULL
  metRecords <- NULL
  for (trial in 1:nTrials){
    trialPval <- runif(1, min=envPvals[trial,1], max=envPvals[trial,2])
    for (rep in 1:nReps){
      phenoV <- AlphaSimR::setPheno(pop, varE=varE, p=trialPval, onlyPheno=T)
      # get the environment-specific genotypic value by setting varE=0
      genoV <- AlphaSimR::setPheno(pop, varE=0, p=trialPval, onlyPheno=T)
      metRecords <- rbind(metRecords,
                          data.frame(paste0("Env", trial),
                                paste0("Rep", rep), pop@id, phenoV, genoV))
    }
  }
  colnames(metRecords) <- c("env", "rep", "id", "phenoVal", "genoVal")
  return(metRecords)
}

### Analyze the MET with lmer.  

In [ ]:
nEnv <- 10
useEnvPvals <- cbind(rep(0.05, nEnv), rep(0.95, nEnv))
metResults <- conductMET(founders, envPvals=useEnvPvals)
metResults$env <- as.factor(metResults$env)
metResults$rep <- as.factor(metResults$rep)
metResults$id <- as.factor(metResults$id)
cs_lmer <- lme4::lmer(phenoVal ~ env + (1 | id) + (1 | id:env), data=metResults)
summary(cs_lmer)

1. As discussed, each environment is characterized by `p`, which determines the environment's loading. In the example, I chose `p` between 0.05 and 0.95. In the AlphaSimR scheme of things, subdividing the environments into sets of similar environments (TPE) is equivalent to choosing `p` within smaller segments along the [0, 1] line. Run 10 replicates and plot histograms or box plots of the genetic variance estimates, GxE variance estimates, and genetic correlation between environments. (2 pts)  
2. Let's look at the impact of narrowing the range of environments and the impact on estimates of variance components. Choose a range of `p` of from 0.4 and 0.6 and "conduct a MET" in that range. Run 10 replicates and plot histograms or box plots of the genetic variance estimates, GxE variance estimates, and genetic correlation between environments. (2 pts)
3. Compare the results from 1 and 2. Provide an explaination of the results. (2 pts)  
4. Now simulate an MET with range for `p` from .75 to .95 (note these environments are very different from the target environment `p`=.5). Run 10 replicates and plot histograms or box plots of the genetic variance estimates, GxE variance estimates, and genetic correlation between environments. (2 pts)  
5. What about the way `AlphaSimR` simulates GxE might be causing this? (1 pt)
6. 1 point for turning in on time.